# Worked Example — Plan a One-Day City Visit

A ReAct agent requests information through tools, receives observations and uses them to decide what to request next or to prepare an answer. The model proposes requests; Python reads the local records and executes the lookup functions. Tool access does not guarantee a feasible plan. We check the result against the original records.

In this project, you will explore four fictional city datasets, connect four lookup tools, write the agent's instructions and examine a one-day itinerary. Begin with a prediction, run the supplied demonstration and check its cost and schedule. Optional local OpenAI mode generates fresh requests; offline mode is fixed author-written teaching text and does not respond to edited prompts. No bookings or payments are made.

Run every cell in order in a fresh kernel. Files stay beside this notebook. Requirements at the companion root include pandas, JupyterLab, openai and python-dotenv. No key is needed offline. All records are synthetic for 15 June 2026; they are not live travel information.

## 1. Understand the request

The visitor wants three history/art attractions between 10:00 and 17:00, admission at most €40, walking between stops. The visit starts at the first attraction and ends at the last. Food, queues and hotel transfers are outside scope. Use the explicitly dated request below.

**Predict before running:** Which records will you need? What could make a plan fail? Write your prediction here.

In [ ]:
question = 'I have one day to explore the city on 2026-06-15, from 10:00 to 17:00.\nI enjoy history and art, and I would like to visit three attractions.\nMy admission budget is €40. I will walk between locations.\nPlease check opening hours and the weather before suggesting a plan.\nThe budget covers admission only. Start at the first attraction and\nfinish at the last; no journeys from a hotel need to be planned.'


## 2. Explore the records

`attractions.csv` gives prices, interests, indoor/outdoor settings and suggested durations. `opening_hours.csv` gives dated open/closed records. `weather.csv` supplies the forecast. `travel_times.csv` gives walking minutes for directed pairs of attraction identifiers. A closed attraction is different from an unavailable record. The prices are admission only, in euros. Inspect the tables before choosing a route.

In [ ]:
import json
import os
from pathlib import Path
from types import SimpleNamespace
import pandas as pd
from IPython.display import display

# Keep the four CSV files beside this notebook.
attractions = pd.read_csv("attractions.csv", keep_default_na=False)
opening_hours = pd.read_csv("opening_hours.csv", keep_default_na=False)
weather = pd.read_csv("weather.csv", keep_default_na=False)
travel_times = pd.read_csv("travel_times.csv", keep_default_na=False)
for name, frame in [("Attractions", attractions), ("Opening hours", opening_hours),
                    ("Weather", weather), ("Walking times", travel_times)]:
    print(name)
    display(frame.head(6))


## 3. Prepare and test the tools

These are the same read-only filtering patterns as Section 5.2, now applied to a different domain. Empty lists mean no matching record, not zero cost or zero journey time. Run a known lookup and one missing lookup. Explain the difference.

In [ ]:
def get_attractions():
    # Return the small catalogue so the model can compare interests and prices.
    return attractions.to_dict(orient="records")


def get_opening_hours(attraction_id, date):
    rows = opening_hours[
        (opening_hours["attraction_id"] == attraction_id)
        & (opening_hours["date"] == date)
    ]
    return rows.to_dict(orient="records")


def get_weather(date):
    return weather[weather["date"] == date].to_dict(orient="records")


def get_travel_time(origin_id, destination_id):
    rows = travel_times[
        (travel_times["origin_id"] == origin_id)
        & (travel_times["destination_id"] == destination_id)
    ]
    return rows.to_dict(orient="records")


In [ ]:
print("Known hours:", get_opening_hours("A01", "2026-06-15"))
print("Unavailable date:", get_opening_hours("A01", "2099-01-01"))
print("Closed attraction:", get_opening_hours("A06", "2026-06-15"))


### Describe the tools

These completed descriptions explain what each tool retrieves.

In [ ]:
# Describe what each function retrieves. These descriptions do not execute it.
tool_descriptions = {
    "get_attractions": "Retrieve attraction identifiers, names, interests, admission prices, visit durations and indoor/outdoor settings.",
    "get_opening_hours": "Retrieve a dated opening record for an attraction. A closed status means do not schedule it; an empty result means no record is available.",
    "get_weather": "Retrieve the supplied weather record for a date; an empty result means unavailable weather information.",
    "get_travel_time": "Retrieve walking minutes from the origin attraction to the destination. An empty result means the journey time is unavailable, not zero."
}

# Tool parameters use the same JSON-schema structure as Section 5.2.
text_parameter = {"type": "string"}
tools = []
for name, parameters in [
    ("get_attractions", {}),
    ("get_opening_hours", {"attraction_id": text_parameter, "date": text_parameter}),
    ("get_weather", {"date": text_parameter}),
    ("get_travel_time", {"origin_id": text_parameter, "destination_id": text_parameter})
]:
    tools.append({
        "type": "function", "name": name,
        "description": tool_descriptions[name],
        "parameters": {
            "type": "object", "properties": parameters,
            "required": list(parameters), "additionalProperties": False
        }
    })


## 4. Write the agent instructions

This worked prompt states the task, output expectations and evidence limits.

In [ ]:
system_prompt = "You are a city-visit planning assistant.\n\nUse the available tools to prepare an itinerary that\nmatches the visitor's interests, time and admission budget.\n\nCheck attraction details, opening hours, weather and\nwalking times before presenting the plan.\n\nInclude:\n- Each attraction and its planned arrival and departure.\n- Walking time between consecutive attractions.\n- Admission charges and their total.\n- A short explanation of how the plan meets the request.\n\nUse only information supported by the returned records.\nDo not invent opening hours, prices or travel times.\nIdentify missing information and explain any assumptions.\n\nIf the request cannot be met, explain which constraint\nprevents it and suggest a clearly labelled alternative."


## 5. Run and inspect the investigation

Offline mode uses seven supplied lookups and one supplied final answer. The CSV functions really execute, but the sequence and answer are fixed; changing instructions or dates does not change that fixture. It is not an evaluation of your prompt.

For live mode, use local Python, install the root requirements and copy `.env.example` to `.env` beside `COMPANION_VERSION`. Set your own `OPENAI_API_KEY` and `OPENAI_MODEL` privately. Complete the TODOs, then set `USE_OPENAI=True`. One investigation allows at most **12 requests**, output capped at 500 tokens per request, retries disabled. Refinement is a separate opt-in run with up to 12 additional requests. Model decisions may vary. Do not rerun live cells simply to redisplay an answer. Browser Python stays offline.

In [ ]:
# Fixed author-written teaching sequence: this is not model reasoning.
def make_supplied_client():
    actions = [
        ("get_attractions", {}),
        ("get_weather", {"date": "2026-06-15"}),
        ("get_opening_hours", {"attraction_id": "A01", "date": "2026-06-15"}),
        ("get_opening_hours", {"attraction_id": "A02", "date": "2026-06-15"}),
        ("get_opening_hours", {"attraction_id": "A03", "date": "2026-06-15"}),
        ("get_travel_time", {"origin_id": "A01", "destination_id": "A02"}),
        ("get_travel_time", {"origin_id": "A02", "destination_id": "A03"})
    ]
    state = {"step": 0}
    def create(**request):
        i = state["step"]
        state["step"] += 1
        if i < len(actions):
            name, arguments = actions[i]
            call = SimpleNamespace(type="function_call", name=name,
                arguments=json.dumps(arguments), call_id=f"supplied-{i}")
            return SimpleNamespace(output=[call], output_text="")
        return SimpleNamespace(output=[], output_text=(
            "SUPPLIED ITINERARY — not a live model response\n"
            "10:00–11:30 City History Museum (€12).\n"
            "11:30–11:50 Walk to Riverside Art Gallery (20 minutes).\n"
            "11:50–13:20 Riverside Art Gallery (€15).\n"
            "13:20–13:35 Walk to Heritage House (15 minutes).\n"
            "13:35–14:35 Heritage House (€8). Total admission: €35.\n"
            "Three indoor visits suit history/art interests and the recorded rain. "
            "Walking is outdoors. The forecast is a teaching record, not a guarantee. "
            "The plan finishes at 14:35; the remaining time is unallocated. "
            "Food, queues and journeys to/from the first/last stop are not included."
        ))
    return SimpleNamespace(responses=SimpleNamespace(create=create), close=lambda: None)


### Enable OpenAI here — local JupyterLab only

The configuration cell immediately below controls the mode. For a fresh model-generated itinerary, open this **worked notebook locally**, install the companion-root `requirements.txt`, and save your own `OPENAI_API_KEY` and `OPENAI_MODEL` in `.env` beside `COMPANION_VERSION`. Keep all four CSVs beside the notebook. The completed prompt and descriptions above are ready to use.

Change `USE_OPENAI = False` to **`USE_OPENAI = True`**. Leave `RUN_REFINEMENT = False` and `MAX_STEPS = 12` for the first run. Run from the top so the data, functions, descriptions, prompt, client factory and loop are defined before the investigation cell.

The investigation prints **LIVE RUN**, then each requested tool, its arguments and the CSV observation. **FINAL ANSWER** is the fresh OpenAI itinerary. A run uses at most 12 requests, with retries disabled. Do not rerun live cells just to redisplay an answer. The optional refinement run is separate and can use up to 12 more requests.

For the supplied demonstration, keep **`USE_OPENAI = False`** and run from the top. The **SUPPLIED ITINERARY** is author-written and fixed; it does not respond to edited prompts. Browser Python must stay offline. Never put your key into a notebook cell or output.


In [ ]:
# Leave both switches False for a normal offline run.
USE_OPENAI = False
RUN_REFINEMENT = False
MAX_STEPS = 12


def create_client():
    if not USE_OPENAI:
        return make_supplied_client(), "supplied-example"

    # Live mode follows Sections 2.3 and 5.3; it requires local Python.
    import sys
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("Keep offline mode in browser Python.")
    from dotenv import load_dotenv
    from openai import OpenAI

    config_folder = Path.cwd()
    for folder in [config_folder, *config_folder.parents]:
        if (folder / "COMPANION_VERSION").is_file():
            config_folder = folder
            break
    load_dotenv(config_folder / ".env")
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        raise RuntimeError("Set OPENAI_API_KEY and OPENAI_MODEL in package-root .env.")
    if "TODO" in system_prompt or any("TODO" in d for d in tool_descriptions.values()):
        raise ValueError("Complete the prompt and tool descriptions before live execution.")
    return OpenAI(timeout=45.0, max_retries=0), os.environ["OPENAI_MODEL"]


### Reuse the Section 5.3 loop

The message handling, one-tool-per-turn setting, bounded loop and stopping behaviour are unchanged. Only the four function branches have changed. `run_investigation()` starts a fresh messages list each time. An early answer may still lack evidence; reaching the cap or returning empty text leaves the task unfinished.

In [ ]:
def run_investigation(client, model, instructions, question, max_steps=12):
    system_prompt = instructions
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        response = client.responses.create(
            model=model,
            instructions=system_prompt,
            input=messages,
            tools=tools,
            parallel_tool_calls=False,
            max_output_tokens=500,
            store=False
        )
    
        # Retain the complete response for the next request.
        messages += response.output
    
        # Select function requests from the response.
        tool_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]
    
        # No tool request: display the model's answer and stop.
        if not tool_calls:
            if response.output_text.strip():
                print("\nFINAL ANSWER")
                print(response.output_text)
            else:
                print("No answer returned. The investigation is unfinished.")
            break
    
        # This example allows one tool request per model turn.
        tool_call = tool_calls[0]
        arguments = json.loads(tool_call.arguments)
    
        print(f"\nStep {step + 1}")
        print("Requested tool:", tool_call.name)
        print("Arguments:", arguments)
    
        # Execute only the functions explicitly available here.
        if tool_call.name == "get_attractions":
            observation = get_attractions()
        elif tool_call.name == "get_opening_hours":
            observation = get_opening_hours(arguments["attraction_id"], arguments["date"])
        elif tool_call.name == "get_weather":
            observation = get_weather(arguments["date"])
        elif tool_call.name == "get_travel_time":
            observation = get_travel_time(arguments["origin_id"], arguments["destination_id"])
        else:
            observation = {"error": "Unknown tool."}

        print("Observation:")
        print(json.dumps(observation, indent=2, ensure_ascii=False))
    
        # Return the result with its corresponding request identifier.
        messages.append({
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(observation)
        })
    
    else:
        print("\nStep limit reached. The investigation is unfinished.")
    return messages


In [ ]:
# One baseline run. Each iteration may request one lookup or a final answer.
client, model = create_client()
try:
    print("LIVE RUN" if USE_OPENAI else "OFFLINE: fixed supplied interaction")
    baseline_messages = run_investigation(
        client, model, system_prompt, question, MAX_STEPS
    )
finally:
    client.close()


### Interpret the OpenAI answer before checking it

Read the **FINAL ANSWER** from the preceding **LIVE RUN**, together with its printed tool observations. The model can choose another valid route or lookup order; it need not reproduce the supplied plan. Check that each chosen attraction has supported prices/durations, dated opening hours, weather information and recorded walks between stops.

The next cost cell starts with the **reference** identifiers `A01`, `A02`, `A03`. If the live answer chose different attractions, replace `selected_ids` with its three identifiers before running that check. Verify the reported times against visit durations, opening hours and walking records too. A matching total alone does not establish feasibility.

The later `worked_plan` and schedule checks verify an **author-written reference**, not a parsed model response. Do not describe those checks as having verified your live itinerary. Compare the actual model schedule separately. Note one supported finding and any missing evidence, impossible timing or invented detail. A step-limit or empty-answer message means the investigation is unfinished.


## 6. Check the itinerary

Compare the final answer with the CSV records. Check each visit duration and opening window, both walking journeys, the start/end times and admission total. Check that the chosen attractions fit history/art interests. Indoor visits can suit the recorded rain, but the walks remain outdoors. Missing travel times must not be invented.

The next calculation checks the supplied demonstration only. Change `selected_ids` for a different live plan; it does not parse or automatically approve the model answer.

In [ ]:
# Read the prices directly from the CSV, independently of the model's arithmetic.
# For a different itinerary, change the identifiers to the chosen attractions.
selected_ids = ["A01", "A02", "A03"]
selected = attractions[attractions["attraction_id"].isin(selected_ids)]
if len(selected) == 3:
    total_admission = selected["admission_eur"].sum()
    print(f"Admission: €{total_admission:.2f}; within €40: {total_admission <= 40}")
else:
    print("A selected attraction is missing; check the identifiers.")


### Inspect a missing record

What should the agent do when the requested date or journey has no record? Explain why an empty list cannot establish a zero-minute walk.

In [ ]:
print("Unknown journey:", get_travel_time("A01", "A99"))
print("Unavailable weather:", get_weather("2099-01-01"))


## 7. Refine one instruction

Identify a weakness before making a change. If the report already meets a requirement, do not add it again. Keep the question, tools, data and model unchanged. Compare fresh runs; no automatic feedback loop is used. Offline users write the change they would test and their expected improvement; enabling the switch offline only repeats the supplied demonstration, not a new decision.

In [ ]:
# Set the switch in the configuration cell only after inspecting the baseline.
# Replace this example with an instruction addressing your observed weakness.
additional_instruction = "Show each walking journey as a separate entry with departure, arrival and recorded duration."
if RUN_REFINEMENT:
    client, model = create_client()
    try:
        revised_messages = run_investigation(
            client, model, system_prompt + "\n" + additional_instruction,
            question, MAX_STEPS
        )
    finally:
        client.close()
else:
    print("No second investigation: inspect the baseline before revising.")


## Save your work

Record your prediction, completed tool descriptions and prompt, actual or supplied requests, itinerary, cost/schedule checks and one strength/limitation. Identify supplied text, your own writing and live output clearly. Save or download the notebook with its four CSVs. The learner-facing worked example is in the separate `05-04-city-visit-walkthrough` folder; consult it after your attempt.

## Complete worked itinerary

This author-written itinerary is checked against the CSVs, not presented as a live OpenAI response. It uses three indoor history/art attractions. Compare this only after your own attempt.

**When using OpenAI:** the table below is still the author-written reference. It does not replace or validate the earlier live answer. Its €35 cost and 14:35 finish apply only to this reference route. Compare your model’s selected attractions and times with the records separately.

In [ ]:
# Author-written reference plan for comparison after your own attempt.
worked_plan = pd.DataFrame([
    {"attraction_id": "A01", "arrival": "10:00", "departure": "11:30"},
    {"attraction_id": "A02", "arrival": "11:50", "departure": "13:20"},
    {"attraction_id": "A03", "arrival": "13:35", "departure": "14:35"}
])
display(worked_plan.merge(attractions, on="attraction_id")[
    ["arrival", "departure", "name", "admission_eur", "visit_minutes"]
])


In [ ]:
# Keep the time check readable: convert HH:MM into minutes after midnight.
def minutes(time_text):
    hour, minute = map(int, time_text.split(":"))
    return hour * 60 + minute

for stop in worked_plan.to_dict(orient="records"):
    attraction = next(a for a in get_attractions()
                      if a["attraction_id"] == stop["attraction_id"])
    hours = get_opening_hours(stop["attraction_id"], "2026-06-15")[0]
    duration = minutes(stop["departure"]) - minutes(stop["arrival"])
    print(attraction["name"])
    print("  Visit duration:", duration, "minutes")
    print("  Enough time:", duration >= attraction["visit_minutes"])
    print("  Within opening hours:", hours["status"] == "open"
          and minutes(hours["opens"]) <= minutes(stop["arrival"])
          and minutes(stop["departure"]) <= minutes(hours["closes"]))

for i in range(len(worked_plan) - 1):
    previous = worked_plan.iloc[i]
    following = worked_plan.iloc[i + 1]
    walking = get_travel_time(previous["attraction_id"], following["attraction_id"])[0]
    gap = minutes(following["arrival"]) - minutes(previous["departure"])
    print("Walking time fits:", gap >= walking["walking_minutes"])
print("Within visitor's day:", minutes(worked_plan.iloc[0]["arrival"]) >= minutes("10:00")
      and minutes(worked_plan.iloc[-1]["departure"]) <= minutes("17:00"))


### Discussion

Admission is €12 + €15 + €8 = €35, leaving €5 of admission budget. Visits take 90 + 90 + 60 = 240 minutes; walking takes 20 + 15 = 35 minutes. From 10:00 to 14:35 is 275 minutes, leaving 145 minutes before 17:00 unallocated. The gallery opens at 11:00 and Heritage House at 13:00, so the planned arrivals fit. The watchtower is closed. The outdoor garden is not required in rainy weather. No queues, meals or journeys beyond the stated boundaries have been invented.

Other plans can be valid; check the same constraints rather than copying this order. A missing walking record needs clarification or another supported route. Refining a prompt can improve a report but is not evidence of fresh model performance in offline mode.